# Recomendador de películas con similitud de coseno

Notebook para crear un sistema recomendador de películas usando similitud del coseno con un dataset de 400 filas, 10 usuarios y 50 películas.

In [ ]:
# Section: Preparar dependencias
import random
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Section: Crear dataset de ratings
users = [f'user{i:02d}' for i in range(1, 11)]
movie_ids = [f'movie{i:02d}' for i in range(1, 51)]
movie_titles = {movie_id: title for movie_id, title in zip(movie_ids, [
    'El misterio del bosque', 'Viaje al planeta perdido', 'La noche del samurái', 'El legado del tesoro',
    'Amor en tiempos de lluvia', 'El último viaje', 'La fórmula secreta', 'Retorno a la ciudad dorada',
    'Ciudad de sombras', 'El compositor olvidado', 'Aventuras en la montaña', 'Los guardianes del tiempo',
    'Fuga en el tren nocturno', 'La isla de cristal', 'El sueño de la actriz', 'Bajo el mismo techo',
    'El código del silencio', 'Misión en la capital', 'El corredor nocturno', 'El pacto de los tres',
    'Sombras bajo el agua', 'El hito final', 'El reino oculto', 'Secretos en la biblioteca',
    'La rebelión del artista', 'Regreso a casa', 'La verdad del detective', 'El campo de fuego',
    'Olas del destino', 'La fiesta de medianoche', 'El legado del pintor', 'El último concierto',
    'La promesa del verano', 'El viaje del inventor', 'El laberinto del recuerdo', 'Destinos cruzados',
    'Canción de la ciudad', 'La leyenda del bosque', 'La sombra del general', 'Costumbres de un pueblo',
    'La herencia de la familia', 'El guardián de la luna', 'La última frontera', 'El código del faro',
    'El baile escondido', 'La cámara secreta', 'Sueño de artista', 'El rescate imposible',
    'La crónica perdida', 'Conexión final'
])}

rows = []
for user_index, user_id in enumerate(users, start=1):
    for movie_index, movie_id in enumerate(movie_ids, start=1):
        overall_index = (user_index - 1) * len(movie_ids) + (movie_index - 1)
        if overall_index % 5 == 0:
            continue
        rating = (user_index * movie_index + 4) % 6
        rows.append({'user_id': user_id, 'movie_id': movie_id, 'rating': rating})

ratings_df = pd.DataFrame(rows)
ratings_df.to_csv(Path(__file__).resolve().parent / 'movie_ratings.csv', index=False, encoding='utf-8')
print('Dataset generado con:', len(ratings_df), 'filas')
ratings_df.head()

# Section: Calcular matriz de similitud de coseno entre usuarios
ratings_pivot = ratings_df.pivot(index='user_id', columns='movie_id', values='rating')
ratings_fill = ratings_pivot.fillna(0.0)
user_similarity = cosine_similarity(ratings_fill.values)
user_similarity_df = pd.DataFrame(user_similarity, index=ratings_fill.index, columns=ratings_fill.index)
print('Matriz de similitud calculada con forma', user_similarity.shape)
user_similarity_df.iloc[:5, :5]

# Section: Construir sistema de recomendación colaborativo
def predict_ratings_for_new_user(new_user_ratings, ratings_pivot):
    movies = ratings_pivot.columns.tolist()
    existing_users = ratings_pivot.index.tolist()
    new_ratings = pd.Series(data=np.nan, index=movies, dtype=float)
    for movie_id, rating in new_user_ratings.items():
        new_ratings.loc[movie_id] = rating

    ratings_matrix = pd.concat([ratings_pivot, new_ratings.to_frame().T], axis=0)
    filled_matrix = ratings_matrix.fillna(0.0)
    similarity = cosine_similarity(filled_matrix.values)
    active_index = len(existing_users)
    sim_with_others = similarity[active_index, :active_index]

    movie_scores = {}
    for movie_id in movies:
        if pd.notna(new_ratings.loc[movie_id]):
            continue
        ratings_for_movie = ratings_pivot[movie_id]
        valid_mask = ratings_for_movie.notna().values
        if not valid_mask.any():
            movie_scores[movie_id] = 0.0
            continue

        weighted_sum = (ratings_for_movie.values[valid_mask] * sim_with_others[valid_mask]).sum()
        weight_total = np.abs(sim_with_others[valid_mask]).sum()
        movie_scores[movie_id] = weighted_sum / weight_total if weight_total > 0 else 0.0

    return pd.Series(movie_scores).sort_values(ascending=False)


def recommend_top_n(predictions, n=10):
    return predictions.head(n)

# Section: Simular interacción en terminal para calificar 5 películas
def ask_for_user_ratings(movie_ids, movie_titles, target_count=5):
    selected_movies = random.sample(movie_ids, len(movie_ids))
    user_ratings = {}

    print('Califica 5 películas al azar. Deja vacío si no la viste. 0 significa que no te gustó.')

    for movie_id in selected_movies:
        if len(user_ratings) >= target_count:
            break

        title = movie_titles[movie_id]
        answer = input(f'Película: "{title}" (0-5, vacío si no la viste): ').strip()
        if answer == '':
            print('Marcado como no vista. Te mostraré otra película.')
            continue
        if not answer.isdigit():
            print('Entrada inválida, ingresa 0-5 o vacío.')
            continue

        value = int(answer)
        if value < 0 or value > 5:
            print('El valor debe estar entre 0 y 5.')
            continue

        user_ratings[movie_id] = value
        print(f'Guardado: {title} = {value}')

    return user_ratings


def main():
    user_ratings = ask_for_user_ratings(movie_ids, movie_titles, target_count=5)
    if len(user_ratings) < 5:
        print('No se completaron las 5 calificaciones. Reinicia para intentarlo nuevamente.')
        return

    predictions = predict_ratings_for_new_user(user_ratings, ratings_pivot)
    recommendations = recommend_top_n(predictions, n=10)

    print('\nPelículas recomendadas para ti:')
    for idx, (movie_id, score) in enumerate(recommendations.items(), start=1):
        print(f'{idx}. {movie_titles[movie_id]} - puntuación estimada: {score:.2f}')


if __name__ == '__main__':
    main()
